# Baseline Model for Binary Classification

Since this is a binary classification problem (predicting `Will_Buy_EV`), we start with **Logistic Regression** as the baseline model. Linear Regression is intended for continuous target variables, whereas Logistic Regression outputs probabilities suitable for classification and AUC ROC evaluation. Once this baseline is established, we can later increase model complexity using models like LightGBM.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# Set paths
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
SUBMISSION_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

In [ ]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

In [ ]:
# Data Preprocessing
# Target variable mapping
target_col = 'Will_Buy_EV'

if train[target_col].dtype == 'object':
    train[target_col] = train[target_col].map({'No': 0, 'Yes': 1})

# Separate features and target
X = train.drop(columns=['id', target_col])
y = train[target_col]
X_test = test.drop(columns=['id'])

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical features:", categorical_cols)
print("Numerical features:", numerical_cols)

In [ ]:
# Preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
# Validation
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)

# Predict probabilities for AUC ROC
y_pred_proba = model.predict_proba(X_valid)[:, 1]

# Evaluate
auc = roc_auc_score(y_valid, y_pred_proba)
print(f"Validation AUC ROC: {auc:.4f}")

In [ ]:
# Retrain on full data and predict on test
model.fit(X, y)

test_pred_proba = model.predict_proba(X_test)[:, 1]

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    target_col: test_pred_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()